# Complainify AI — 09 : Sentiment Dataset Authoring

Companion notebook to the main *Complainify_AI_Study.ipynb*.
> **Convention (v6):** `negative` now requires STRONG complaint language. Mild problem-reports are labeled `neutral`; the priority column stays the authoring's original urgency labels (orthogonal to sentiment).


---
## 09: Sentiment & Priority Dataset Authoring

This notebook documents how the **12,000-row** sentiment/priority dataset was built: hand-written curation (source=`claude`) plus **template-driven synthesis** (`source=synth`). Every row carries *ground-truth* tags by construction — sentiment and priority are decided at generation time, independently of any classifier.

In [1]:
import os, sys, json, csv, math, random
from collections import Counter, defaultdict
BASE = r'E:\Project-VI\workspace\ComplaintMgmtSystem'
sys.path.insert(0, os.path.join(BASE, 'ml'))
random.seed(42)
DATA = os.path.join(BASE, 'data', 'sentiment_dataset.csv')
rows = list(csv.DictReader(open(DATA, encoding='utf-8')))
print('total rows   :', len(rows))
print('per source   :', dict(Counter(r['source'] for r in rows)))
print('per sentiment:', dict(Counter(r['sentiment_label'] for r in rows)))
print('per priority :', dict(Counter(r['priority'] for r in rows)))
print('per category :', dict(Counter(r['category'] for r in rows)))

total rows   : 12000
per source   : {'claude': 186, 'synth': 11814}
per sentiment: {'positive': 4000, 'negative': 4000, 'neutral': 4000}
per priority : {'Low': 6003, 'Medium': 3940, 'High': 2057}
per category : {'Library': 1215, 'Hostel/Accommodation': 1212, 'Administrative': 1179, 'General/Suggestion': 1223, 'Infrastructure': 1200, 'Examination': 1191, 'Academic': 1183, 'IT/Technical': 1205, 'Faculty/Staff Behavior': 1206, 'Canteen/Food': 1186}


### Authoring ground truth

Priority is assigned by a **deterministic, text-only rule** (`assign_priority` in the generator):

| sentiment | rule |
|---|---|
| negative | High if a severity/urgency phrase appears (e.g. `has never been repaired`, `nothing has been done yet`), else Medium |
| neutral | Medium if the topic carries a deadline/window/timeline/form/schedule noun, else Low |
| positive | always Low — thank-you content is never promoted |

The High/Medium phrase pools are class-locked inside each generated sentence, so the marker vocabulary in the text is exactly the label. The held-out test split is therefore pure and learnable — no row derives from a classifier's output.


In [2]:
import os, sys, json, csv, math, random
from collections import Counter, defaultdict
BASE = r'E:\Project-VI\workspace\ComplaintMgmtSystem'
sys.path.insert(0, os.path.join(BASE, 'ml'))
random.seed(42)
import statistics
rows = list(csv.DictReader(open(os.path.join(BASE, 'data', 'sentiment_dataset.csv'), encoding='utf-8')))
lens = [len(r['complaint_text']) for r in rows]
print('length min/median/max :', min(lens), int(statistics.median(lens)), max(lens))
uniq = len({r['complaint_text'] for r in rows})
print('unique texts          :', uniq, '({:.1f}%)'.format(uniq / len(rows) * 100))
print('missing sentiment     :', sum(1 for r in rows if not r['sentiment_label']))
print('bad priority values   :', sum(1 for r in rows if r['priority'] not in ('Low', 'Medium', 'High')))
print('empty categories      :', sum(1 for r in rows if not r['category']))
print()
print('sample rows:')
for r in random.sample(rows, 5):
    print('  [{:<8}|{:<6}] {}'.format(r['sentiment_label'], r['priority'], r['complaint_text'][:80]))

length min/median/max : 43 92 182
unique texts          : 12000 (100.0%)
missing sentiment     : 0
bad priority values   : 0
empty categories      : 0

sample rows:
  [positive|Low   ] It truly helps that the friendly admission responses we really valued the suppor
  [positive|Low   ] Really appreciate that the supportive mentoring approach – thanks to the concern
  [neutral |Low   ] May I know the current status of the licence approval process If possible, share
  [neutral |Medium] Seeking clarification on the fee refund timeline Are walk ins accepted?
  [positive|Low   ] Refreshing to note that the polite caretaker team we really valued the support.


### Quality gates

The generator validates before every batch is written: priority stays in `{Low, Medium, High}`, sentiment stays in `{positive, neutral, negative}`, texts are ordinary complaint sentences, and the hand-written `source=claude` rows are kept untouched. The CSV is the *single source of truth* for the training script and the web app alike.